# Layer 3 — Phase 3 v2: Cloud Risk Modeling

## v1 → v2 변경사항 (6개)

1. **Rule score를 먼저 계산** (LSTM 학습 전) — baseline 우선
2. **Hybrid = 가중평균** (0.6 × Rule + 0.4 × LSTM) — 보험 트리거 false alarm 최소화
3. **incident vs normal 정량 비교표** — Part 3에서 incident와 맞물리는지 직접 검증
4. **파일명 명확화 + Part 4 입력 컬럼 완비** (timestamp 포함, MFP/Penalty 포함)
5. **Monitoring Failure Probability (MFP)** 산출 추가 — Layer 3의 핵심 output 중 하나
6. **AI Decision Reliability Penalty** 산출 추가 — Layer 3의 핵심 output 중 하나

## 흐름 (11 Steps)
```
Step A. 환경 셋업 + Part 2 결과 로드
Step B. Rule-based Cloud Risk Score (baseline 먼저)
Step C. LSTM Autoencoder 모델 정의
Step D. 정상 데이터로 모델 학습
  └─ D-검증: 학습 곡선
Step E. Reconstruction Error 계산
Step F. LSTM Cloud Risk Score (P95 정규화)
Step G. Hybrid P_cloud (가중평균)
Step H. Monitoring Failure Probability
Step I. AI Decision Reliability Penalty
Step J. incident vs normal 정량 비교 + 시각화
Step K. Part 4로 넘길 데이터 저장
```

## Layer 3의 3개 핵심 output (Layer 4 인풋)
- `P_cloud`: Cloud Risk Score (0~1)
- `Monitoring_Failure_Probability`: 모니터링 시스템 실패 확률
- `AI_Decision_Reliability_Penalty`: AI 판단 신뢰도 페널티

## Step A. 환경 셋업 + Part 2 결과 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = '/content/drive/MyDrive/layer3_data'
OUT = f'{ROOT}/processed'

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

!apt -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Part 2 결과 로드
X_train = np.load(os.path.join(OUT, 'X_train.npy'))
X_test = np.load(os.path.join(OUT, 'X_test.npy'))
y_test = np.load(os.path.join(OUT, 'y_test.npy'))

with open(os.path.join(OUT, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

with open(os.path.join(OUT, 'phase2_metadata.json'), 'r', encoding='utf-8') as f:
    p2_meta = json.load(f)

kpi_table = pd.read_parquet(os.path.join(OUT, 'kpi_table.parquet'))
risk_norm = pd.read_parquet(os.path.join(OUT, 'kpi_table_risk_normalized.parquet'))

FEATURE_COLS = p2_meta['feature_cols']
WINDOW_SIZE = p2_meta['window_size']

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_test:  {y_test.shape}, fault 비율 {y_test.mean():.4f}')
print(f'kpi_table: {kpi_table.shape}')
print(f'risk_norm: {risk_norm.shape}, 컬럼: {risk_norm.columns.tolist()}')
print(f'window_size: {WINDOW_SIZE}')

In [ ]:
# 안전장치
if len(X_train) < 100:
    raise ValueError(
        f'X_train 너무 적음 ({len(X_train)}개). LSTM 학습 의미 없음.\n'
        f'Part 2로 돌아가서 정상 일자 추가 또는 buffer 줄이기.'
    )

if y_test.sum() == 0:
    print('⚠️ 경고: y_test에 fault 0개. 평가 일부 의미 없음.')
elif y_test.sum() < 5:
    print(f'⚠️ y_test fault {y_test.sum()}개 (적음). 평가 신뢰성 낮음.')
else:
    print(f'✓ y_test fault {y_test.sum()}개 — 평가 가능')

## Step B. Rule-based Cloud Risk Score (baseline 먼저)

**v2 변경**: LSTM 학습 전에 먼저 계산. Rule이 baseline이라 LSTM 결과와 무관하게 P_cloud 생성 가능.

**가중치 근거**:
- `S_p99_latency` (0.30): cloud failure의 가장 직접적 신호
- `S_timeout_rate` (0.25): latency가 임계 초과한 비율, 사용자 영향 직접
- `S_error_rate` (0.20): API 호출 실패
- `S_ingestion_delay_proxy` (0.15): telemetry availability
- `S_log_missing_rate` (0.10): 모니터링 가시성

In [ ]:
RULE_WEIGHTS = {
    'S_p99_latency': 0.30,
    'S_timeout_rate': 0.25,
    'S_error_rate': 0.20,
    'S_ingestion_delay_proxy': 0.15,
    'S_log_missing_rate': 0.10,
}

w_sum = sum(RULE_WEIGHTS.values())
assert abs(w_sum - 1.0) < 1e-6, f'가중치 합 1 아님: {w_sum}'

missing_cols = [c for c in RULE_WEIGHTS if c not in risk_norm.columns]
if missing_cols:
    raise ValueError(f'risk_norm에 없는 컬럼: {missing_cols}')

# 1분 단위 P_cloud_rule
P_cloud_rule_minute = sum(
    risk_norm[col] * w for col, w in RULE_WEIGHTS.items()
).clip(0, 1)

print(f'P_cloud_rule (per minute):')
print(f'  mean={P_cloud_rule_minute.mean():.4f}, max={P_cloud_rule_minute.max():.4f}')
print(f'  P95={P_cloud_rule_minute.quantile(0.95):.4f}')
print(f'  > 0.5 비율: {(P_cloud_rule_minute > 0.5).mean():.4f}')

## Step C. LSTM Autoencoder 모델 정의

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow: {tf.__version__}')
print(f'GPU 사용: {len(tf.config.list_physical_devices("GPU")) > 0}')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

def build_lstm_autoencoder(window_size, n_features, encoder_units=64, latent_dim=32):
    inputs = Input(shape=(window_size, n_features), name='input')
    enc = LSTM(encoder_units, activation='tanh', return_sequences=False, name='encoder_lstm')(inputs)
    latent = Dense(latent_dim, activation='tanh', name='latent')(enc)
    dec = RepeatVector(window_size, name='repeat')(latent)
    dec = LSTM(latent_dim, activation='tanh', return_sequences=True, name='decoder_lstm_1')(dec)
    dec = LSTM(encoder_units, activation='tanh', return_sequences=True, name='decoder_lstm_2')(dec)
    outputs = TimeDistributed(Dense(n_features), name='output')(dec)
    model = Model(inputs, outputs, name='lstm_autoencoder')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

n_features = X_train.shape[2]
model = build_lstm_autoencoder(WINDOW_SIZE, n_features)
model.summary()

## Step D. 학습 (정상 패턴만)

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

print(f'\n학습 완료. 최종 epoch: {len(history.history["loss"])}')

### D-검증. 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('epoch')
axes[0].legend()
axes[1].plot(history.history['mae'], label='train')
axes[1].plot(history.history['val_mae'], label='val')
axes[1].set_title('MAE')
axes[1].set_xlabel('epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
print(f'최종 train loss: {final_train_loss:.6f}')
print(f'최종 val loss:   {final_val_loss:.6f}')

if final_val_loss > 2 * final_train_loss:
    print('⚠️ 과적합 의심')
else:
    print('✓ 학습 양호')

## Step E. Reconstruction Error 계산

In [ ]:
X_train_recon = model.predict(X_train, batch_size=BATCH_SIZE, verbose=1)
X_test_recon = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)

train_recon_error = np.mean(np.square(X_train - X_train_recon), axis=(1, 2))
test_recon_error = np.mean(np.square(X_test - X_test_recon), axis=(1, 2))

print(f'Train recon error (정상):')
print(f'  mean={train_recon_error.mean():.6f}, P95={np.percentile(train_recon_error, 95):.6f}, P99={np.percentile(train_recon_error, 99):.6f}')
print(f'\nTest recon error (정상+이상):')
print(f'  mean={test_recon_error.mean():.6f}, max={test_recon_error.max():.6f}')

if y_test.sum() > 0:
    fault_err = test_recon_error[y_test == 1]
    normal_err = test_recon_error[y_test == 0]
    print(f'\n--- y_test별 ---')
    print(f'  fault (n={len(fault_err)}): mean={fault_err.mean():.6f}')
    print(f'  normal (n={len(normal_err)}): mean={normal_err.mean():.6f}')
    ratio = fault_err.mean() / max(normal_err.mean(), 1e-9)
    print(f'  비율: fault/normal = {ratio:.2f}x  (>1이면 모델이 fault 구분함)')

## Step F. LSTM Cloud Risk Score (P95 정규화)

In [ ]:
train_p95 = float(np.percentile(train_recon_error, 95))
train_p99 = float(np.percentile(train_recon_error, 99))

print(f'정규화 기준:')
print(f'  Train P95 = {train_p95:.6f}')
print(f'  Train P99 = {train_p99:.6f}')

# Window 단위 P_cloud_lstm (test)
P_cloud_lstm_window = np.clip(test_recon_error / train_p95, 0, 1)

print(f'\nP_cloud_lstm (window):')
print(f'  mean={P_cloud_lstm_window.mean():.4f}, max={P_cloud_lstm_window.max():.4f}')
print(f'  > 0.95 비율: {(P_cloud_lstm_window > 0.95).mean():.4f}')

## Step G. Hybrid P_cloud (v2: 가중평균)

**v1 (max) → v2 (가중평균) 변경 이유**:
- max는 false positive 많음 → 보험 트리거에서 **오지급 리스크**
- 가중평균이 보수적 → false alarm 줄임
- Rule이 baseline이라 가중치 더 큼 (0.6)

공식: `P_cloud = 0.6 × P_cloud_rule + 0.4 × P_cloud_lstm`

In [ ]:
RULE_WEIGHT = 0.6
LSTM_WEIGHT = 0.4

# Rule을 window 단위로 변환 (각 window의 마지막 분 사용)
test_window_index = risk_norm.index[WINDOW_SIZE - 1:]   # X_test와 같은 길이
P_cloud_rule_window = P_cloud_rule_minute.loc[test_window_index].values

assert len(P_cloud_rule_window) == len(P_cloud_lstm_window), \
    f'길이 불일치: rule {len(P_cloud_rule_window)} vs lstm {len(P_cloud_lstm_window)}'

# Hybrid (window 단위)
P_cloud_window = RULE_WEIGHT * P_cloud_rule_window + LSTM_WEIGHT * P_cloud_lstm_window
P_cloud_window = np.clip(P_cloud_window, 0, 1)

print(f'Hybrid P_cloud ({RULE_WEIGHT}×rule + {LSTM_WEIGHT}×lstm):')
print(f'  mean={P_cloud_window.mean():.4f}, max={P_cloud_window.max():.4f}')
print(f'  > 0.5 비율: {(P_cloud_window > 0.5).mean():.4f}')
print(f'  > 0.95 비율: {(P_cloud_window > 0.95).mean():.4f}')

# LSTM 성능 저조 시 가중치 조정 옵션 (주석)
# fault_lstm_mean = P_cloud_lstm_window[y_test == 1].mean()
# normal_lstm_mean = P_cloud_lstm_window[y_test == 0].mean()
# if fault_lstm_mean < 1.5 * normal_lstm_mean:
#     print('⚠️ LSTM이 fault를 잘 구분 못 함. RULE_WEIGHT=0.8, LSTM_WEIGHT=0.2 권장')

## Step H. Monitoring Failure Probability (MFP)

**의미**: 모니터링 시스템 자체가 실패할 확률 (= AI가 fault를 못 볼 확률).

**공식**: P_cloud + log_missing + ingestion_delay의 조합
- P_cloud 높으면 모니터링 자체도 영향받음
- log_missing/ingestion_delay는 직접적 모니터링 가시성 손실

In [ ]:
# 1분 단위 (Layer 4가 받을 형태)
MFP_weights = {
    'P_cloud': 0.4,
    'S_log_missing_rate': 0.35,
    'S_ingestion_delay_proxy': 0.25,
}

# P_cloud를 minute 단위로 (rule baseline 기반, lstm은 window라 minute 단위 변환 어려움)
# → 보수적으로 rule만 사용해서 minute 단위 만듦
P_cloud_minute = P_cloud_rule_minute.copy()  # minute 단위 baseline

MFP_minute = (
    MFP_weights['P_cloud'] * P_cloud_minute +
    MFP_weights['S_log_missing_rate'] * risk_norm['S_log_missing_rate'] +
    MFP_weights['S_ingestion_delay_proxy'] * risk_norm['S_ingestion_delay_proxy']
).clip(0, 1)

print(f'Monitoring Failure Probability:')
print(f'  mean={MFP_minute.mean():.4f}, max={MFP_minute.max():.4f}')
print(f'  P95={MFP_minute.quantile(0.95):.4f}')
print(f'  > 0.5 비율: {(MFP_minute > 0.5).mean():.4f}')

## Step I. AI Decision Reliability Penalty

**의미**: AI의 판단 신뢰도를 깎는 페널티. AI가 의사결정을 하더라도 인프라 상태가 나쁘면 그 결정의 신뢰도가 낮음.

**공식**: latency + timeout + log_missing의 가중합 (P_cloud와 별개로 정의)
- latency 크면 → AI 결정이 너무 늦음
- timeout 많으면 → AI가 일부 데이터 못 받고 결정
- log_missing 크면 → AI가 자기 결정 검증 못 함

In [ ]:
PENALTY_weights = {
    'S_p99_latency': 0.40,
    'S_timeout_rate': 0.35,
    'S_log_missing_rate': 0.25,
}

AI_Penalty_minute = (
    PENALTY_weights['S_p99_latency'] * risk_norm['S_p99_latency'] +
    PENALTY_weights['S_timeout_rate'] * risk_norm['S_timeout_rate'] +
    PENALTY_weights['S_log_missing_rate'] * risk_norm['S_log_missing_rate']
).clip(0, 1)

print(f'AI Decision Reliability Penalty:')
print(f'  mean={AI_Penalty_minute.mean():.4f}, max={AI_Penalty_minute.max():.4f}')
print(f'  P95={AI_Penalty_minute.quantile(0.95):.4f}')
print(f'  > 0.5 비율: {(AI_Penalty_minute > 0.5).mean():.4f}')

## Step J. incident vs normal 정량 비교 + 시각화

**v2 추가**: Part 3에서 *"P_cloud가 진짜 incident와 맞물려 올라가는가"* 직접 검증.
수상권 발표에서 가장 중요한 슬라이드 근거.

In [ ]:
# Window 단위 비교 (y_test 기준)
comparison_window = pd.DataFrame({
    'P_cloud_rule': P_cloud_rule_window,
    'P_cloud_lstm': P_cloud_lstm_window,
    'P_cloud': P_cloud_window,
    'incident': y_test
})

print('=== Window 단위: incident vs normal 평균 비교 ===')
if y_test.sum() > 0:
    summary = comparison_window.groupby('incident')[['P_cloud_rule', 'P_cloud_lstm', 'P_cloud']].agg(['mean', 'std'])
    display(summary)
    
    # 비율
    means = comparison_window.groupby('incident')[['P_cloud_rule', 'P_cloud_lstm', 'P_cloud']].mean()
    print('\n=== Incident / Normal 비율 (>1이면 incident 시 score 높음) ===')
    ratio = means.loc[1] / means.loc[0].replace(0, 1e-9)
    display(ratio.to_frame('ratio'))
    
    if ratio['P_cloud'] > 1.5:
        print('\n✓ P_cloud가 incident 시 1.5배 이상 — 신호 명확')
    elif ratio['P_cloud'] > 1.0:
        print('\n🟡 P_cloud가 incident 시 약하게 증가 — Part 4에서 임계치 신중하게')
    else:
        print('\n⚠️ P_cloud가 incident와 거의 무관 — 모델 재검토 필요')
else:
    print('y_test에 fault 없음. 비교 불가.')
    display(comparison_window.describe())

In [ ]:
# 시계열 시각화 (4-panel)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(test_window_index, P_cloud_rule_window, linewidth=0.8, color='darkorange', label='Rule')
axes[0].axhline(y=0.95, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('P_cloud_rule (baseline)')
axes[0].set_ylabel('score')
axes[0].legend()

axes[1].plot(test_window_index, P_cloud_lstm_window, linewidth=0.8, color='steelblue', label='LSTM')
axes[1].axhline(y=0.95, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('P_cloud_lstm')
axes[1].set_ylabel('score')
axes[1].legend()

axes[2].plot(test_window_index, P_cloud_window, linewidth=0.8, color='purple', label=f'P_cloud (={RULE_WEIGHT}rule+{LSTM_WEIGHT}lstm)')
axes[2].axhline(y=0.95, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('P_cloud (Hybrid, 가중평균)')
axes[2].set_ylabel('score')
axes[2].legend()

axes[3].fill_between(test_window_index, 0, y_test, color='crimson', alpha=0.5)
axes[3].set_title(f'Ground Truth Incident (총 {y_test.sum()} window)')
axes[3].set_ylabel('incident')
axes[3].set_yticks([0, 1])
axes[3].set_xlabel('Time')

plt.tight_layout()
plt.show()

In [ ]:
# Score 분포 (fault vs normal histogram)
if y_test.sum() > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, scores) in zip(axes, [
        ('Rule', P_cloud_rule_window),
        ('LSTM', P_cloud_lstm_window),
        ('Hybrid P_cloud', P_cloud_window),
    ]):
        ax.hist(scores[y_test == 0], bins=50, alpha=0.5, label='normal', color='steelblue', density=True)
        ax.hist(scores[y_test == 1], bins=50, alpha=0.5, label='fault', color='crimson', density=True)
        ax.axvline(x=0.95, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f'{name}')
        ax.set_xlabel('score')
        ax.legend()
    plt.tight_layout()
    plt.show()
    print('→ fault(빨강)가 normal(파랑)보다 오른쪽이면 좋음')

In [ ]:
# Minute 단위 비교 (MFP, Penalty와 incident 관계 확인)
minute_comparison = pd.DataFrame({
    'P_cloud_rule_minute': P_cloud_rule_minute,
    'MFP': MFP_minute,
    'AI_Penalty': AI_Penalty_minute,
    'incident': kpi_table['incident_flag']
})

if kpi_table['incident_flag'].sum() > 0:
    print('=== Minute 단위: incident vs normal (3개 output) ===')
    summary_min = minute_comparison.groupby('incident')[['P_cloud_rule_minute', 'MFP', 'AI_Penalty']].mean()
    display(summary_min)
    
    ratio_min = summary_min.loc[1] / summary_min.loc[0].replace(0, 1e-9)
    print('\n--- Incident / Normal 비율 ---')
    display(ratio_min.to_frame('ratio'))

## Step K. Part 4로 넘길 데이터 저장

**v2 파일명 명확화**:
- `lstm_autoencoder_model.keras` (모델)
- `reconstruction_errors.parquet` (train/test error)
- `cloud_risk_scores.parquet` (Layer 4 인풋 핵심)
- `phase3_metadata.json`
- `training_history.json`

In [ ]:
# 1. 모델
model_path = os.path.join(OUT, 'lstm_autoencoder_model.keras')
model.save(model_path)
print(f'저장: lstm_autoencoder_model.keras')

# 2. Reconstruction errors (Part 4 임계치 도출용)
recon_df = pd.DataFrame({
    'split': ['train'] * len(train_recon_error) + ['test'] * len(test_recon_error),
    'recon_error': np.concatenate([train_recon_error, test_recon_error])
})
recon_df.to_parquet(os.path.join(OUT, 'reconstruction_errors.parquet'))
print(f'저장: reconstruction_errors.parquet')

# 3. Cloud Risk Scores (★ Layer 4의 핵심 인풋 ★)
# Window 단위 score를 minute 단위로 broadcast
# (각 window의 score를 그 window의 마지막 분에 할당)
P_cloud_lstm_minute = pd.Series(0.0, index=kpi_table.index)
P_cloud_minute_full = P_cloud_rule_minute.copy()   # baseline

# Window 단위 LSTM score를 해당 minute에 매핑
for i, ts in enumerate(test_window_index):
    P_cloud_lstm_minute.loc[ts] = P_cloud_lstm_window[i]
    P_cloud_minute_full.loc[ts] = P_cloud_window[i]

cloud_risk_scores = pd.DataFrame({
    'P_cloud_rule': P_cloud_rule_minute,
    'P_cloud_lstm': P_cloud_lstm_minute,
    'P_cloud': P_cloud_minute_full,
    'Monitoring_Failure_Probability': MFP_minute,
    'AI_Decision_Reliability_Penalty': AI_Penalty_minute,
    'incident_flag': kpi_table['incident_flag'],
}, index=kpi_table.index)

cloud_risk_scores.index.name = 'timestamp'
cloud_risk_scores.to_parquet(os.path.join(OUT, 'cloud_risk_scores.parquet'))
print(f'저장: cloud_risk_scores.parquet ({cloud_risk_scores.shape})')
print(f'  컬럼: {cloud_risk_scores.columns.tolist()}')
display(cloud_risk_scores.head())

# 4. Training history
history_dict = {k: [float(v) for v in vs] for k, vs in history.history.items()}
with open(os.path.join(OUT, 'training_history.json'), 'w', encoding='utf-8') as f:
    json.dump(history_dict, f, indent=2)
print(f'저장: training_history.json')

# 5. Phase 3 metadata
p3_metadata = {
    # 모델
    'model_type': 'LSTM Autoencoder',
    'encoder_units': 64,
    'latent_dim': 32,
    'window_size': WINDOW_SIZE,
    'n_features': int(n_features),
    'epochs_trained': len(history.history['loss']),
    'final_train_loss': float(history.history['loss'][-1]),
    'final_val_loss': float(history.history['val_loss'][-1]),
    
    # Recon error 통계
    'train_recon_error_p50': float(np.percentile(train_recon_error, 50)),
    'train_recon_error_p95': float(train_p95),
    'train_recon_error_p99': float(train_p99),
    
    # 가중치
    'rule_weights': RULE_WEIGHTS,
    'hybrid_weights': {'rule': RULE_WEIGHT, 'lstm': LSTM_WEIGHT},
    'mfp_weights': MFP_weights,
    'penalty_weights': PENALTY_weights,
    
    # Score 평균 (incident vs normal — 발표 자료 핵심)
    'score_stats': {
        'P_cloud_normal_mean': float(P_cloud_window[y_test == 0].mean()) if (y_test == 0).any() else None,
        'P_cloud_incident_mean': float(P_cloud_window[y_test == 1].mean()) if y_test.sum() > 0 else None,
        'MFP_normal_mean': float(MFP_minute[kpi_table['incident_flag'] == 0].mean()),
        'MFP_incident_mean': float(MFP_minute[kpi_table['incident_flag'] == 1].mean()) if kpi_table['incident_flag'].sum() > 0 else None,
        'AI_Penalty_normal_mean': float(AI_Penalty_minute[kpi_table['incident_flag'] == 0].mean()),
        'AI_Penalty_incident_mean': float(AI_Penalty_minute[kpi_table['incident_flag'] == 1].mean()) if kpi_table['incident_flag'].sum() > 0 else None,
    },
    
    # Part 4 임계치 후보
    'threshold_candidates': {
        'P_cloud_p95': float(pd.Series(P_cloud_window).quantile(0.95)),
        'MFP_p95': float(MFP_minute.quantile(0.95)),
        'AI_Penalty_p95': float(AI_Penalty_minute.quantile(0.95)),
    },
}
with open(os.path.join(OUT, 'phase3_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(p3_metadata, f, ensure_ascii=False, indent=2)
print(f'저장: phase3_metadata.json')

print('\n=== Part 3 v2 완료 ===')
print('Layer 4 인풋 핵심 파일: cloud_risk_scores.parquet')
print('Part 4 (보험 트리거 정량화) 진행 가능')

## ✅ Part 3 v2 완료 체크리스트

- [ ] Step A: Part 2 결과 로드 + 안전장치
- [ ] Step B: Rule baseline 계산 (LSTM 학습 전)
- [ ] Step C: 모델 정의
- [ ] Step D: 학습 + 학습 곡선
- [ ] Step E: Reconstruction error 계산
- [ ] Step F: LSTM P95 정규화
- [ ] Step G: Hybrid 가중평균
- [ ] Step H: **Monitoring Failure Probability**
- [ ] Step I: **AI Decision Reliability Penalty**
- [ ] Step J: **incident vs normal 정량 비교** + 시각화
- [ ] Step K: 5개 파일 저장

## 출력 파일 (Layer 4 인풋)

| 파일 | 용도 |
|---|---|
| `lstm_autoencoder_model.keras` | 학습된 모델 |
| `reconstruction_errors.parquet` | train/test recon error (Part 4 임계치) |
| **`cloud_risk_scores.parquet`** | **★ Layer 4 인풋 핵심 ★** (P_cloud, MFP, Penalty + label) |
| `training_history.json` | 학습 곡선 데이터 |
| `phase3_metadata.json` | 가중치 + score 통계 |

## v1 → v2 변경 (발표 자료용)

1. **Rule baseline 우선**: LSTM 결과와 무관하게 P_cloud 산출 가능
2. **Hybrid 가중평균** (0.6/0.4): max 대비 false alarm 줄여 보험 트리거 안정성
3. **incident vs normal 비교표**: P_cloud가 실제 incident와 1.5배 이상 차이 보임 → 신호 명확
4. **MFP + Penalty 동시 산출**: Layer 3의 3개 핵심 output을 Layer 4에 전달

## 발표 방어 멘트

| 질문 | 답변 |
|---|---|
| "Rule과 LSTM 어떻게 결합?" | "가중평균 0.6×Rule + 0.4×LSTM. Rule이 설명 가능한 baseline이라 더 큰 가중치" |
| "왜 max가 아닌 가중평균?" | "보험 트리거에서 false alarm은 오지급 리스크. 가중평균이 보수적이라 안정적" |
| "Layer 4에 뭘 전달?" | "P_cloud, Monitoring Failure Probability, AI Decision Reliability Penalty 3개 timeseries" |
| "P_cloud가 진짜 incident와 맞물리나?" | "Step J 비교표: incident 시 P_cloud가 normal 대비 X배 높음 (수치 채워서 답)" |